In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score


In [2]:
df_dataset = pd.read_parquet("../data/processed/dataset_features.parquet")

df_dataset = df_dataset[df_dataset["condition"].isin(["EO", "EC"])]

df_dataset["y"] = df_dataset["condition"].map({"EO": 0, "EC": 1})

In [4]:
NODE_FEATURES = [
    c for c in df_dataset.columns
    if c.startswith(("degree_", "clustering_", "betweenness_", "hub_"))
]

In [5]:
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [6]:
META_FEATURES = ["age", "gender"]

In [7]:
def evaluate_model(X, y, model):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "acc": "accuracy",
        "auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    acc_mean = scores["test_acc"].mean()
    acc_std = scores["test_acc"].std()

    auc_mean = scores["test_auc"].mean()
    auc_std = scores["test_auc"].std()

    print(
        "=== Model Evaluation (Stratified 5-Fold CV) ===\n"
        f"Mean Accuracy : {acc_mean:.3f} ± {acc_std:.3f}\n"
        f"Mean ROC-AUC  : {auc_mean:.3f} ± {auc_std:.3f}\n"
        "Note: ROC-AUC values close to 0.5 indicate random performance."
    )

    return {
        "accuracy_mean": acc_mean,
        "auc_mean": auc_mean
    }


A  função acima treina e avalia um modelo de classificação usando validação cruzada estratificada com 5 folds, calcula acurácia e AUC em cada fold e retorna a média dessas métricas.

### Regressão Logística

In [9]:
X = df_dataset[NODE_FEATURES].select_dtypes(include="number").dropna(axis=1)
y = df_dataset["y"]

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.421 ± 0.024
Mean ROC-AUC  : 0.422 ± 0.064
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': 0.4208333333333333, 'auc_mean': 0.4222886029411764}

Avaliação do modelo por validação cruzada estratificada (5 folds).
O modelo apresentou acurácia média de 42,1% e AUC média de 0,42, indicando um desempenho inferior ao esperado para um classificador binário.

A acurácia próxima a 50% sugere que o modelo possui baixa capacidade preditiva, enquanto o valor de AUC abaixo de 0,5 indica que o modelo não consegue separar adequadamente as classes, apresentando desempenho próximo — ou até inferior — ao de um classificador aleatório.

Esses resultados indicam a necessidade de revisão das features, engenharia de atributos, ajuste de hiperparâmetros ou a avaliação de modelos mais complexos.

### Regressão Logística com PCA

In [10]:

pipe_pca_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_pca_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.463 ± 0.059
Mean ROC-AUC  : 0.453 ± 0.087
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': 0.46344696969696975, 'auc_mean': 0.4530790441176471}

Os modelos de classificação supervisionada baseados em regressão logística apresentaram desempenho próximo ao aleatório (AUC ≈ 0,45), mesmo após a aplicação de PCA. Esse resultado sugere que, em uma população saudável, as métricas globais e locais de conectividade funcional não são suficientes para discriminar automaticamente as condições analisadas, reforçando a elevada variabilidade interindividual típica de cérebros saudáveis.

### Regressão Logística com PCA e metadados

In [13]:
X_meta = pd.concat(
    [X, df_dataset[META_FEATURES]],
    axis=1
).dropna()

y_meta = df_dataset.loc[X_meta.index, "y"]

pipe_pca_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X_meta, y_meta, pipe_pca_meta)


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Anne\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Anne\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Anne\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 469, in fit
    Xt = self._fit(X, y, routed_params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Anne\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 406, in _fit
    X, fitted_transformer = fit